In [ ]:
pip install -q accelerate peft transformers datasets bitsandbytes trl torch

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

# --- Konfigurasi Model dan Tokenizer ---
model_name = "meta-llama/Llama-2-7b-chat-hf"  # Ganti dengan model yang kamu miliki

# Konfigurasi Quantization 4-bit (untuk menghemat memori)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Muat model dalam mode 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,  # Diperlukan untuk gradient checkpointing
)

# Muat tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# --- Siapkan Dataset ---
# Muat dataset dari file JSON lokal
dataset = load_dataset('json', data_files='data/my_training_data.json', split='train')

# --- Konfigurasi LoRA ---
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # Modul yang akan dilatih
)

# Siapkan model untuk training k-bit (quantized)
model = prepare_model_for_kbit_training(model)

# Terapkan adapter LoRA ke model
model = get_peft_model(model, peft_config)

# --- Konfigurasi Training Arguments ---
training_arguments = TrainingArguments(
    output_dir="./finetuned_lora_adapter",  # <--- Folder hasil fine-tuning
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    warmup_ratio=0.03,
    num_train_epochs=3,
    save_steps=100,
    save_total_limit=3,
)

# --- Inisialisasi Trainer dan Mulai Training ---
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_arguments,
    train_dataset=dataset,
    max_seq_length=512,
    dataset_text_field="output",  # Kolom yang akan dijadikan target pembelajaran
)

# Mulai proses fine-tuning
trainer.train()

# --- Simpan Model Hasil Fine-Tuning ---
trainer.model.save_pretrained("./finetuned_lora_adapter")
trainer.tokenizer.save_pretrained("./finetuned_lora_adapter")
print("Fine-tuning selesai! Model adapter disimpan di './finetuned_lora_adapter'")